In [2]:
import pandas as pd

excel_path = r"C:\Projects\Manuela Del Castillo\Data\ICSR_assessments_16092025.xlsx"
txt_path   = r"C:\Projects\Manuela Del Castillo\Data\ICSR_assessments_16092025.txt"

# Read the Excel sheet
df = pd.read_excel(excel_path, sheet_name="ICSR_assessment", header=0)

# Save as tab-delimited text
df.to_csv(txt_path, sep="\t", index=False)

print(f"✅ Saved tab-delimited TXT at: {txt_path}")




✅ Saved tab-delimited TXT at: C:\Projects\Manuela Del Castillo\Data\ICSR_assessments_16092025.txt


Exception during reset or similar
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\mysql\connector\connection_cext.py", line 772, in cmd_query
    self._cmysql.query(
_mysql_connector.MySQLInterfaceError: Got a packet bigger than 'max_allowed_packet' bytes

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\sqlalchemy\engine\base.py", line 1936, in _exec_single_context
    self.dialect.do_executemany(
  File "C:\ProgramData\anaconda3\Lib\site-packages\sqlalchemy\engine\default.py", line 938, in do_executemany
    cursor.executemany(statement, parameters)
  File "C:\ProgramData\anaconda3\Lib\site-packages\mysql\connector\cursor_cext.py", line 473, in executemany
    return self.execute(cast(str, stmt))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\mysql\connector\cursor_cext.py", line 353, in execute
 

MySQLInterfaceError: Lost connection to MySQL server during query

In [3]:
from sqlalchemy import create_engine, text
from sqlalchemy.types import Text, String

# -----------------------------
# 1) Path to your TXT file
# -----------------------------
txt_path = r"C:\Projects\Manuela Del Castillo\Data\ICSR_assessments_16092025.txt"

# -----------------------------
# 2) Load the tab-delimited TXT
# -----------------------------
df = pd.read_csv(txt_path, sep="\t")

# -----------------------------
# 3) Clean column names for MySQL compatibility
# -----------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-zA-Z_]+', '_', regex=True)
      .str.replace(r'__+', '_', regex=True)
      .str.strip('_')
)

# -----------------------------
# 4) Connect to MySQL (XAMPP on port 3307)
# -----------------------------
engine = create_engine("mysql+mysqlconnector://root:@127.0.0.1:3307/db_icsr_assessment_manuela")

# -----------------------------
# 5) Define column types (long text columns → TEXT)
# -----------------------------
dtype_map = {}
for col in df.columns:
    if 'narrative' in col or 'reasoning' in col or 'text' in col:
        dtype_map[col] = Text()
    elif col.endswith('_id'):
        dtype_map[col] = String(64)

# -----------------------------
# 6) Write DataFrame to SQL in chunks
# -----------------------------
table_name = "icsr_assessment_import"

df.to_sql(
    name=table_name,
    con=engine,
    if_exists="replace",   # ⚠️ overwrite table if exists; use 'append' later
    index=False,
    dtype=dtype_map,
    chunksize=500          # ✅ insert in batches of 500 rows
)

# -----------------------------
# 7) Verify row count
# -----------------------------
with engine.connect() as conn:
    rowcount = conn.execute(text(f"SELECT COUNT(*) FROM {table_name}")).scalar()
    print(f"✅ Loaded {rowcount} rows into {table_name}.")


✅ Loaded 780 rows into icsr_assessment_import.
